# Specific Test IV — Neural Operator Classifier
## Fourier Neural Operator (FNO2d) for Gravitational Lens Substructure Classification

**Backbone:** 2-D Fourier Neural Operator  
**Dataset:** Same as Common Test I — 30k train / 7.5k val, 150×150 grayscale `.npy`  
**Input:** Same 3-channel physics representation (raw + gradient magnitude + Laplacian)  
**Evaluation:** ROC AUC one-vs-rest macro average

| Property | ConvNeXt V2 baseline | FNO (this work) |
|---|---|---|
| Receptive field | Local (kernel-bounded) | **Global** (all Fourier modes couple everywhere) |
| Weight sharing | Spatial pixel-space | **Spectral frequency-space** |
| Resolution invariance | Fixed 224×224 | Resolution-independent |
| Physics alignment | Generic | Spectral ops align with PDE-generated lensing fields |
| Pretraining | ImageNet-1k | None — learns from lensing data only |

In [1]:
import numpy as np
import os
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from torchvision import transforms
import random
import torch.nn as nn
import torch.nn.functional as F
import gc

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
torch.cuda.empty_cache(); gc.collect()

Device: cuda


17

## 1. Dataset — Identical to Common Test I

Same `LensDataset`, same physics channels, same computed statistics. The only variable changed is the model architecture.

In [2]:
class LensDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.paths, self.labels = [], []
        self.classes = sorted([d for d in os.listdir(root_dir)
                                if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        for cls in self.classes:
            cls_folder = os.path.join(root_dir, cls)
            for file in os.listdir(cls_folder):
                if file.endswith(".npy"):
                    self.paths.append(os.path.join(cls_folder, file))
                    self.labels.append(self.class_to_idx[cls])

    def __len__(self): return len(self.paths)

    def gradient_magnitude(self, img):
        eps = 1e-8; I = img + eps
        gx = torch.gradient(I, dim=-1)[0]; gy = torch.gradient(I, dim=-2)[0]
        mag = torch.sqrt(gx**2 + gy**2 + eps)
        return (mag - mag.min()) / (mag.max() - mag.min() + eps)

    def laplacian_channel(self, img):
        eps = 1e-8; I = img + eps
        gxx = torch.gradient(torch.gradient(I, dim=-1)[0], dim=-1)[0]
        gyy = torch.gradient(torch.gradient(I, dim=-2)[0], dim=-2)[0]
        return torch.abs(torch.tanh(gxx + gyy))

    def __getitem__(self, idx):
        image = torch.tensor(np.load(self.paths[idx]), dtype=torch.float32)
        if image.ndim == 2: image = image.unsqueeze(0)
        combined = torch.cat([image,
                               self.gradient_magnitude(image),
                               self.laplacian_channel(image)], dim=0)
        if self.transform: combined = self.transform(combined)
        return combined, self.labels[idx]

In [ ]:
# Channel statistics from Common Test I (no recomputation — no data leakage)
MEAN = [0.0617, 0.0589, 0.0069]
STD  = [max(s, 0.0005) for s in [0.1173, 0.1013, 0.0099]]

# FNO operates natively at 150x150 — no upsampling to 224 needed
IMG_SIZE = 150

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180, fill=0),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), fill=0),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.08), ratio=(0.3, 3.3)),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Normalize(mean=MEAN, std=STD)
])

train_dataset = LensDataset("dtrainataset/", transform=train_transform)
val_dataset   = LensDataset("dataset/val",   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Input shape: {train_dataset[0][0].shape}")  # (3, 150, 150)

FileNotFoundError: [Errno 2] No such file or directory: 'Common Test 1/dataset/train'

## 2. Fourier Neural Operator Architecture

### SpectralConv2d — the FNO core layer

Every FNO block applies a **2D spectral convolution**:

```
x (B,C,H,W)
  → rfft2        complex spectrum (B, C, H, W//2+1)
  → truncate     keep lowest modes1 × modes2 frequencies  ← physics-informed gate
  → learned ℂ weights  R ∈ ℂ^{C_in × C_out × modes1 × modes2}
  → pad + irfft2 spatial field (B, C_out, H, W)
```

**Key difference from CNN:** Each weight is in *frequency space* — it couples a **global Fourier mode** to every spatial location simultaneously. Effective receptive field = entire image, from layer 1.

### Mode selection strategy

| modes | Coverage on 150px | What it captures |
|---|---|---|
| 4 | ~2.7% | Only coarsest global shape |
| 12 | ~8% | Large-scale ring position |
| **20** | **~13%** | **Arc morphology, ring perturbations ← chosen** |
| 40 | ~27% | Approaches full convolution |

Modes 4–20 cover the frequency bands where lensing substructure is physically present. Higher modes (> 30) are dominated by noise — deliberately excluded.

In [5]:
class SpectralConv2d(nn.Module):
    """
    2D Fourier layer — operates in function/frequency space.

    Unlike CNN kernels that slide locally, each weight here couples a GLOBAL
    Fourier mode. Effective receptive field = full image from layer 1.
    Physically aligned with PDE-generated lensing fields.
    """
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.out_channels = out_channels
        self.modes1, self.modes2 = modes1, modes2
        scale = 1.0 / (in_channels * out_channels)
        # real/imag stored separately for full torch.compile compatibility
        self.wr1 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2))
        self.wi1 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2))
        self.wr2 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2))
        self.wi2 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2))

    def _cmul(self, xr, xi, wr, wi):
        return (torch.einsum("bixy,ioxy->boxy", xr, wr) - torch.einsum("bixy,ioxy->boxy", xi, wi),
                torch.einsum("bixy,ioxy->boxy", xr, wi) + torch.einsum("bixy,ioxy->boxy", xi, wr))

    def forward(self, x):
        B, C, H, W = x.shape
        ft = torch.fft.rfft2(x, norm="ortho")
        fr, fi = ft.real, ft.imag
        or_ = torch.zeros(B, self.out_channels, H, W//2+1, device=x.device)
        oi  = torch.zeros_like(or_)
        r1, i1 = self._cmul(fr[:,:,:self.modes1,:self.modes2], fi[:,:,:self.modes1,:self.modes2], self.wr1, self.wi1)
        or_[:,:,:self.modes1,:self.modes2] = r1; oi[:,:,:self.modes1,:self.modes2] = i1
        r2, i2 = self._cmul(fr[:,:,-self.modes1:,:self.modes2], fi[:,:,-self.modes1:,:self.modes2], self.wr2, self.wi2)
        or_[:,:,-self.modes1:,:self.modes2] = r2; oi[:,:,-self.modes1:,:self.modes2] = i2
        return torch.fft.irfft2(torch.complex(or_, oi), s=(H, W), norm="ortho")


class FNOBlock2d(nn.Module):
    """
    FNO residual block: SpectralConv2d(x) + Conv1x1(x) -> InstanceNorm -> GELU

    Conv1x1 bypass:
    - Restores fine spatial detail discarded by frequency truncation
    - Gradient highway for stable deep training (analogous to ResNet shortcuts)
    """
    def __init__(self, channels, modes1, modes2):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes1, modes2)
        self.bypass   = nn.Conv2d(channels, channels, 1)
        self.norm     = nn.InstanceNorm2d(channels, affine=True)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.bypass(x)))


class FNOClassifier(nn.Module):
    """
    FNO-based classifier for gravitational lensing substructure.

    Accepts the SAME (B, 3, 150, 150) physics-channel input as the ConvNeXt baseline.
    Replaces the convolutional feature extractor with FNO blocks operating in function space.
    """
    def __init__(self, in_channels=3, width=64, modes1=20, modes2=20,
                 depth=4, num_classes=3, dropout=0.3):
        super().__init__()
        # lift input channels to latent width
        self.lift = nn.Sequential(
            nn.Conv2d(in_channels, width, 3, padding=1),
            nn.InstanceNorm2d(width, affine=True), nn.GELU())
        # FNO blocks — learn in frequency space
        self.fno_blocks = nn.ModuleList([FNOBlock2d(width, modes1, modes2) for _ in range(depth)])
        # project to 2x channels
        proj_ch = width * 2
        self.project = nn.Sequential(
            nn.Conv2d(width, proj_ch, 1), nn.BatchNorm2d(proj_ch), nn.GELU())
        # dual pooling: avg captures global ring appearance, max captures salient arcs
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)
        # classification head
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(proj_ch*2, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes))

    def forward(self, x):
        x = self.lift(x)
        for blk in self.fno_blocks: x = blk(x)
        x = self.project(x)
        return self.head(torch.cat([self.gap(x), self.gmp(x)], dim=1))

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# verify
model = FNOClassifier()
dummy = torch.randn(2, 3, 150, 150)
print(f"Output shape: {model(dummy).shape}")  # (2, 3)
print(f"Parameters : {model.count_parameters():,}")
print(f"ConvNeXt V2: ~28,000,000")

Output shape: torch.Size([2, 3])
Parameters : 26,309,123
ConvNeXt V2: ~28,000,000


## 3. Training

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate(model, loader, device):
    """Identical to Common Test I baseline — fair comparison guaranteed."""
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item()
            all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    p = np.concatenate(all_probs); l = np.concatenate(all_labels)
    return total_loss/len(loader), (p.argmax(1)==l).mean(), roc_auc_score(l, p, multi_class='ovr', average='macro')


def train_fno(model, train_loader, val_loader, device,
              epochs=50, lr=3e-3, weight_decay=1e-4,
              patience=10, save_path='best_fno.pth'):

    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    # OneCycleLR: 10% warm-up + cosine decay
    # FNO spectral weights start near zero -> gradual warm-up is critical
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader),
        pct_start=0.1, anneal_strategy='cos', div_factor=10.0, final_div_factor=1e3)
    scaler = torch.amp.GradScaler()

    history = {k:[] for k in ['train_loss','val_loss','train_auc','val_auc','train_acc','val_acc','lr']}
    best_auc = best_epoch = patience_counter = 0

    print("="*75)
    print(f"FNO Classifier | {epochs} epochs | lr={lr} | {model.count_parameters():,} params")
    print("="*75)
    print(f"{'Ep':>4}  {'T-Loss':>7}  {'V-Loss':>7}  {'T-AUC':>7}  {'V-AUC':>7}  {'T-Acc':>6}  {'V-Acc':>6}")
    print("-"*75)

    for epoch in range(1, epochs+1):
        model.train()
        tl, tp, tlab = 0.0, [], []
        for imgs, labs in train_loader:
            imgs, labs = imgs.to(device), labs.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=device.type):
                loss = criterion(model(imgs), labs)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            tl += loss.item()
            with torch.no_grad():
                tp.append(F.softmax(model(imgs), dim=1).cpu().numpy())
            tlab.append(labs.cpu().numpy())

        tp = np.concatenate(tp); tlab = np.concatenate(tlab)
        t_auc = roc_auc_score(tlab, tp, multi_class='ovr', average='macro')
        t_acc = (tp.argmax(1)==tlab).mean()
        vl, v_acc, v_auc = evaluate(model, val_loader, device)

        for k,v in zip(['train_loss','val_loss','train_auc','val_auc','train_acc','val_acc','lr'],
                       [tl/len(train_loader), vl, t_auc, v_auc, t_acc, v_acc, optimizer.param_groups[0]['lr']]):
            history[k].append(v)

        flag = " ✓" if v_auc > best_auc else ""
        print(f"{epoch:>4}  {tl/len(train_loader):>7.4f}  {vl:>7.4f}  {t_auc:>7.4f}  {v_auc:>7.4f}  {t_acc:>6.2%}  {v_acc:>6.2%}{flag}")

        if v_auc > best_auc:
            best_auc = v_auc; best_epoch = epoch; patience_counter = 0
            torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'val_auc':best_auc,'history':history}, save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping (best epoch {best_epoch}, AUC {best_auc:.4f})")
                break

    print(f"\nBest val AUC: {best_auc:.4f} at epoch {best_epoch}")
    return history

In [ ]:
torch.cuda.empty_cache(); gc.collect()

model = FNOClassifier(
    in_channels=3, width=64, modes1=20, modes2=20,
    depth=4, num_classes=3, dropout=0.3
).to(DEVICE)

history = train_fno(
    model, train_loader, val_loader, DEVICE,
    epochs=50, lr=3e-3, weight_decay=1e-4,
    patience=10, save_path='best_fno.pth'
)

## 4. Final Evaluation

In [ ]:
ckpt = torch.load('best_fno.pth', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded epoch {ckpt['epoch']} — Val AUC: {ckpt['val_auc']:.4f}")

val_loss, val_acc, val_auc = evaluate(model, val_loader, DEVICE)
print(f"\nFinal Val AUC : {val_auc:.4f}")
print(f"Final Val Acc : {val_acc:.4f}")
print(f"Final Val Loss: {val_loss:.4f}")

## 5. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc as sklearn_auc
from sklearn.preprocessing import label_binarize

model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labs in val_loader:
        all_probs.append(F.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy())
        all_labels.append(labs.numpy())

all_probs  = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
class_names = ['no_sub', 'subhalo', 'vortex']
labels_bin  = label_binarize(all_labels, classes=[0,1,2])

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'coral', 'seagreen']
aucs = []
for i, (name, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(labels_bin[:,i], all_probs[:,i])
    roc_auc = sklearn_auc(fpr, tpr); aucs.append(roc_auc)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f"{name}  (AUC = {roc_auc:.4f})")

macro_auc = np.mean(aucs)
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate',  fontsize=13)
ax.set_title(f'ROC Curves — FNO Classifier   (Macro AUC = {macro_auc:.4f})', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()
# plt.savefig('figures/fno_roc.png', dpi=150)
plt.show()

print(f"Macro AUC (FNO)       : {macro_auc:.4f}")
print(f"Macro AUC (ConvNeXt)  : 0.9910")
print(f"Delta                 : {macro_auc - 0.9910:+.4f}")

## 6. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

preds = all_probs.argmax(axis=1)
cm    = confusion_matrix(all_labels, preds)
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — FNO Classifier')
plt.tight_layout(); plt.show()
print(f"Val Accuracy: {(preds==all_labels).mean():.4f}")

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
ep = range(1, len(history['train_loss'])+1)
axes[0].plot(ep, history['train_loss'], label='Train'); axes[0].plot(ep, history['val_loss'], label='Val')
axes[0].set(title='Loss', xlabel='Epoch'); axes[0].legend()
axes[1].plot(ep, history['train_auc'], label='Train'); axes[1].plot(ep, history['val_auc'], label='Val')
axes[1].set(title='Macro AUC', xlabel='Epoch'); axes[1].legend()
axes[2].plot(ep, history['lr']); axes[2].set(title='LR (OneCycleLR)', xlabel='Epoch', yscale='log')
plt.suptitle('FNO Classifier — Training Curves', fontsize=13)
plt.tight_layout(); plt.show()

## 8. Test-Time Augmentation (TTA)

In [ ]:
def evaluate_tta(model, loader, device):
    """8-transform TTA: 4 rotations x 2 flips. Identical to Common Test I baseline."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            imgs = imgs.to(device)
            bp = torch.zeros(imgs.shape[0], 3).to(device)
            for angle in [0, 90, 180, 270]:
                r = transforms.functional.rotate(imgs, angle)
                bp += F.softmax(model(r), dim=1)
                bp += F.softmax(model(transforms.functional.hflip(r)), dim=1)
            bp /= 8
            all_probs.append(bp.cpu().numpy()); all_labels.append(labs.numpy())
    p = np.concatenate(all_probs); l = np.concatenate(all_labels)
    auc = roc_auc_score(l, p, multi_class='ovr', average='macro')
    acc = (p.argmax(1)==l).mean()
    print(f"TTA Val AUC: {auc:.4f}  |  TTA Val Acc: {acc:.4f}")
    return acc, auc

tta_acc, tta_auc = evaluate_tta(model, val_loader, DEVICE)

## 9. Spectral Feature Visualization

Unique to FNO — we can directly inspect what the model learned **in Fourier space**. This is impossible with a CNN.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
class_names_d = ['no_sub', 'subhalo', 'vortex']

model.eval()
with torch.no_grad():
    # Row 1: log|FFT| of raw channel per class
    for c in range(3):
        idx = val_dataset.labels.index(c)
        img, _ = val_dataset[idx]
        mag = torch.log1p(torch.fft.rfft2(img[0], norm="ortho").abs()).numpy()
        axes[0][c].imshow(mag, cmap='plasma')
        axes[0][c].set_title(f'log|FFT| — {class_names_d[c]}', fontsize=10)
        axes[0][c].axis('off')

    # Row 1, col 4: learned weight magnitudes in block 1
    sp = model.fno_blocks[0].spectral
    wm = torch.sqrt(sp.wr1**2 + sp.wi1**2).mean(dim=(0,1)).numpy()
    im = axes[0][3].imshow(wm, cmap='viridis', aspect='auto')
    axes[0][3].set_title('Block 1: learned |W|\nin Fourier space', fontsize=10)
    plt.colorbar(im, ax=axes[0][3], fraction=0.046)

    # Row 2: learned weights across all 4 blocks
    for b in range(4):
        sp = model.fno_blocks[b].spectral
        wm = torch.sqrt(sp.wr1**2 + sp.wi1**2).mean(dim=(0,1)).numpy()
        im = axes[1][b].imshow(wm, cmap='viridis', aspect='auto')
        axes[1][b].set_title(f'Block {b+1}: learned |W|', fontsize=10)
        plt.colorbar(im, ax=axes[1][b], fraction=0.046)

plt.suptitle('Fourier space analysis — input spectra and learned FNO weights', fontsize=12)
plt.tight_layout()
# plt.savefig('figures/fno_spectral_viz.png', dpi=150)
plt.show()
print("Bright = Fourier modes weighted most heavily by the FNO for classification.")

## 10. Comparison Summary

In [ ]:
_, fno_val_acc, fno_val_auc = evaluate(model, val_loader, DEVICE)

BASELINE_AUC = 0.9910
BASELINE_ACC = 0.9910

print("=" * 62)
print(f"  {'Metric':<25} {'ConvNeXt V2':>12} {'FNO':>12} {'Delta':>8}")
print("-" * 62)
print(f"  {'Val Macro AUC':<25} {BASELINE_AUC:>12.4f} {fno_val_auc:>12.4f} {fno_val_auc-BASELINE_AUC:>+8.4f}")
print(f"  {'Val Accuracy':<25} {BASELINE_ACC:>12.4f} {fno_val_acc:>12.4f} {fno_val_acc-BASELINE_ACC:>+8.4f}")
print(f"  {'Parameters':<25} {'~28M':>12} {str(model.count_parameters()//1000)+'K':>12}")
print(f"  {'Input resolution':<25} {'224x224':>12} {'150x150':>12}")
print(f"  {'ImageNet pretrain':<25} {'Yes':>12} {'No':>12}")
print(f"  {'Receptive field':<25} {'Local (conv)':>12} {'Global (FFT)':>12}")
print(f"  {'Operates in':<25} {'Pixel space':>12} {'Freq space':>12}")
print("=" * 62)

## 11. Discussion

### How FNO differs from standard CNNs

**ConvNeXt V2 (baseline):**  Local depthwise convolutions, kernel-bounded receptive field, ImageNet-pretrained weights in pixel space.

**FNO (this work):**  Spectral convolutions via FFT. From the very first layer, every spatial position is coupled to every other through Fourier modes. Weights are learned in frequency space — each weight governs a specific oscillation frequency globally across the image.

### Why this architecture makes physical sense

Gravitational lensing images are the output of a PDE (the lens equation). Each substructure type leaves a characteristic spectral signature:

| Class | Physical process | Spectral signature |
|---|---|---|
| no_sub | Smooth mass distribution | Power concentrated in low-freq ring modes, circular symmetry |
| subhalo | Localized mass concentration | Mid-freq perturbation at arc location, breaks ring symmetry locally |
| vortex | Distributed ring perturbation | Angular power spectrum asymmetry — *fundamentally a global feature* |

The vortex class in particular is a **distributed angular perturbation** of the Einstein ring. This is a global feature that a CNN can only capture after stacking many layers to build up global context. FNO captures it in a single spectral convolution.

### Why no ImageNet pretraining is needed

CNNs benefit from ImageNet pretraining because they learn general-purpose edge detectors and texture representations that transfer well across visual domains. FNO operates in frequency space — ImageNet-pretrained frequency representations would not transfer to lensing physics in any useful way. Training from scratch on lensing data is the correct approach for FNO.

### Strategy summary

1. **Mode selection:** 20×20 on 150px images — covers ~13% of spectrum, targeting arc morphology and ring perturbation frequencies  
2. **Same physics channels:** raw + gradient magnitude + Laplacian — validated in Common Test I EDA  
3. **OneCycleLR from scratch:** FNO spectral weights initialise near zero; warm-up is critical  
4. **Dual pooling:** avg pool (global ring appearance) + max pool (salient arc perturbations)  
5. **Depth 4:** four spectral convolutions progressively refine the frequency-space representation